## Improving the flight price model

> Paste each "# %%" block into its own cell, at the END of your notebook
> (after the Kaggle submission section or before it, your choice).
> It re-loads the raw data so it can use the `flight` column that the notebook drops.

In [8]:
import sklearn
print(sklearn.__version__)
print(sklearn.__file__)

1.0.2
c:\Users\pc\anaconda3\lib\site-packages\sklearn\__init__.py


In [9]:
# [1] Rebuild the data with the extra features
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

raw = pd.read_csv('data/train.csv')

# same cleaning as your notebook
cols_no_id = [c for c in raw.columns if c != 'id']
raw = raw.drop_duplicates(subset=cols_no_id)
raw = raw[raw['price'] > 0]
for c in ['airline', 'source_city', 'destination_city', 'departure_time', 'arrival_time', 'stops', 'flight_class']:
    vc = raw[c].value_counts()
    raw = raw[~raw[c].isin(vc[vc < 30].index)]
raw = raw.reset_index(drop=True)


def build_features(d, flight_counts=None):
    d = d.copy()
    out = pd.DataFrame(index=d.index)
    out['flight_class'] = d['flight_class'].map({'Economy': 0, 'Business': 1})
    out['stops'] = d['stops'].map({'zero': 0, 'one': 1, 'two_or_more': 2})
    out['duration'] = d['duration']
    out['days_left'] = d['days_left']
    out['dep_time'] = d['departure_time'].map(
        {'Early_Morning': 0, 'Morning': 1, 'Afternoon': 2, 'Evening': 3, 'Night': 4, 'Late_Night': 5})
    out['arr_time'] = d['arrival_time'].map(
        {'Early_Morning': 0, 'Morning': 1, 'Afternoon': 2, 'Evening': 3, 'Night': 4, 'Late_Night': 5})

    # NEW: route, flight number pieces
    out['route'] = (d['source_city'] + '>' + d['destination_city'])
    out['airline'] = d['airline']
    out['source_city'] = d['source_city']
    out['destination_city'] = d['destination_city']
    out['flight'] = d['flight']                                   # high-cardinality id, handled below
    out['flight_num'] = pd.to_numeric(d['flight'].str.extract(r'(\d+)')[0], errors='coerce').fillna(0)

    # NEW: simple interactions / transforms that trees find hard to build themselves
    out['class_x_days'] = out['flight_class'] * d['days_left']
    out['log_days_left'] = np.log1p(d['days_left'])
    out['urgent'] = (d['days_left'] <= 7).astype(int)
    out['class_x_stops'] = out['flight_class'] * out['stops']
    out['dur_per_stop'] = d['duration'] / (1 + out['stops'])
    return out


X_all = build_features(raw)
y_all = raw['price'].values
print(X_all.shape)

(18038, 17)


In [10]:
# [2] Preprocessor: ordinal-encode categoricals + cross-fitted target encoding of flight / route
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, TargetEncoder

low_card = ['airline', 'source_city', 'destination_city', 'route']
high_card = ['flight']
numeric = [c for c in X_all.columns if c not in low_card + high_card]

def make_pre():
    # TargetEncoder cross-fits internally during fit_transform, so there is no target leakage.
    # NOTE: we fit it on log(price) because TransformedTargetRegressor passes the logged target down.
    return ColumnTransformer([
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), low_card),
        ('te',  TargetEncoder(target_type='continuous', smooth='auto', random_state=101), high_card + ['route']),
        ('num', 'passthrough', numeric),
    ])

cat_mask = [True] * len(low_card) + [False] * (1 + 1 + len(numeric))   # only the ordinal block is categorical

ImportError: cannot import name 'TargetEncoder' from 'sklearn.preprocessing' (c:\Users\pc\anaconda3\lib\site-packages\sklearn\preprocessing\__init__.py)

In [ ]:
# [3] Gradient boosting (the usual big win on tabular data) vs your tree
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor

xtr, xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.25, random_state=101)

def log_wrap(model):
    return TransformedTargetRegressor(Pipeline([('pre', make_pre()), ('m', model)]),
                                      func=np.log1p, inverse_func=np.expm1)

models = {
    'HistGB (log)': log_wrap(HistGradientBoostingRegressor(
        categorical_features=cat_mask, learning_rate=0.05, max_iter=600,
        max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.1, random_state=101)),
    'RandomForest (log)': log_wrap(RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2, max_features=0.5, n_jobs=-1, random_state=101)),
    'ExtraTrees (log)': log_wrap(ExtraTreesRegressor(
        n_estimators=500, min_samples_leaf=2, max_features=0.7, n_jobs=-1, random_state=101)),
}

preds, rows = {}, []
for name, m in models.items():
    m.fit(xtr, ytr)
    p = np.clip(m.predict(xte), 1, None)
    preds[name] = p
    rows.append({'model': name,
                 'RMSE': np.sqrt(mean_squared_error(yte, p)),
                 'MAE': mean_absolute_error(yte, p),
                 'R2': r2_score(yte, p)})

# simple average of the three (blending usually beats each member)
blend = np.mean(list(preds.values()), axis=0)
rows.append({'model': 'Blend (mean of 3)', 'RMSE': np.sqrt(mean_squared_error(yte, blend)),
             'MAE': mean_absolute_error(yte, blend), 'R2': r2_score(yte, blend)})
pd.DataFrame(rows).sort_values('RMSE').round(3)

In [ ]:
# [4] Which new features helped? (permutation importance on the best single model)
from sklearn.inspection import permutation_importance
best = models['HistGB (log)']
pi = permutation_importance(best, xte, yte, n_repeats=5, random_state=101,
                            scoring='neg_root_mean_squared_error', n_jobs=-1)
pd.Series(pi.importances_mean, index=xte.columns).sort_values(ascending=False).round(2)

In [ ]:
# [5] Honest check: 5-fold cross-validation of the winners
from sklearn.model_selection import cross_val_score
kf = KFold(n_splits=5, shuffle=True, random_state=101)
for name in ['HistGB (log)', 'RandomForest (log)']:
    s = -cross_val_score(models[name], X_all, y_all, cv=kf,
                         scoring='neg_root_mean_squared_error', n_jobs=1)
    print(f'{name:20s} CV RMSE = {s.mean():.2f} +/- {s.std():.2f}')
    r = cross_val_score(models[name], X_all, y_all, cv=kf, scoring='r2', n_jobs=1)
    print(f'{"":20s} CV R2   = {r.mean():.3f}')

In [ ]:
# [6] (Optional) tune HistGB a little
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'regressor__m__learning_rate':     [0.02, 0.03, 0.05, 0.08],
    'regressor__m__max_leaf_nodes':    [15, 31, 63],
    'regressor__m__min_samples_leaf':  [10, 20, 40],
    'regressor__m__l2_regularization': [0, 1, 5, 10],
}
search = RandomizedSearchCV(models['HistGB (log)'], param_dist, n_iter=20, cv=3,
                            scoring='neg_root_mean_squared_error', random_state=101, n_jobs=-1)
search.fit(xtr, ytr)
print('best params:', search.best_params_)
p = np.clip(search.predict(xte), 1, None)
print('tuned HistGB  RMSE=%.2f  MAE=%.2f  R2=%.3f' % (
    np.sqrt(mean_squared_error(yte, p)), mean_absolute_error(yte, p), r2_score(yte, p)))

In [ ]:
# [7] Predict test.csv with the tuned model (re-train on ALL rows)
import os
if os.path.exists('data/test.csv'):
    test_raw = pd.read_csv('data/test.csv')
    X_test = build_features(test_raw)
    final = search.best_estimator_.fit(X_all, y_all)
    out = pd.DataFrame({'id': test_raw['id'], 'price': np.clip(final.predict(X_test), 1, None)})
    out.to_csv('data/submission_v2.csv', index=False)
    print('saved', out.shape)